# Infosys Quarterly Report Analysis

#### Developed By: Manaranjan Pradhan
#### www.manaranjanp.com

*This Jupyter notebook is confidential and proprietary to Manaranjan Pradhan. It is intended solely for authorized training purposes. Unauthorized distribution, sharing, or reproduction of this notebook or its contents is strictly prohibited. This material is for personal learning within the training program only and may not be used for commercial purposes or shared with others. Unauthorized use may result in disciplinary action or legal consequences. If you have received this notebook without authorization, please contact manaranjan@gmail.com immediately and delete all copies.*

In [1]:
!pip install getpass4 pypdf openai faiss-cpu llama-index llama-index-readers-file llama-index-vector-stores-faiss llama-index-embeddings-openai llama-index-llms-groq llama-index-embeddings-huggingface

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of llama-index-llms-openai-like to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 52.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 63.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 20.3 MB/s  0:00:00
  Created wheel for clipboard: filename=clipboard-0.0.4-py3-none-any.whl size=1902 sha256=441db99b93953e13106c6b0b381ea1105e3dfdd5c2ad2cc33cab5160f928befb
  Stored in directory: /root/.cache/pip/wheels/78/a1/7e/9e3b57761786b38955477687177941eeb279174f18e2e90e5b
Successfully built clipboard
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
   

In [ ]:
#!pip install docling

In [1]:
import nest_asyncio

nest_asyncio.apply()

In [2]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [3]:
from llama_index.core import SimpleDirectoryReader, ServiceContext, VectorStoreIndex, StorageContext
from llama_index.core.response.pprint_utils import pprint_response
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core.node_parser import (SentenceWindowNodeParser,)
from llama_index.core.text_splitter import SentenceSplitter
from llama_index.core import Document
import faiss
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [4]:
import faiss

In [8]:
from docling.document_converter import DocumentConverter

In [9]:
converter = DocumentConverter()
result = converter.convert("/content/ifrs-inr-press-release.pdf")
result.document.export_to_markdown()

# Define the output directory
output_dir = "output_documents"
os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists

# Define output file path
output_path = os.path.join(output_dir, "converted_document.md")

# Save as Markdown
with open(output_path, "w", encoding="utf-8") as f:
    f.write(result.document.export_to_markdown())

print(f"Document saved at: {output_path}")


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv2d(


Document saved at: output_documents/converted_document.md


## Configure LLM service

In [5]:
#llm = OpenAI(temperature=0,
#             model="chatgpt-4o-latest",
#             max_tokens=500)

llm = Groq(model="openai/gpt-oss-120b")

In [6]:
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Configuring for logging

In [59]:
import logging, sys
logging.basicConfig(stream=sys.stdout, level=logging.DEBUG, force=True)
logging.getLogger("httpx").setLevel(logging.DEBUG)   # shows the raw request body

from llama_index.core import set_global_handler
set_global_handler("simple")

In [60]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)
Settings.num_output = 512
Settings.context_window = 4800

## Load data
Downloaded from

https://www.infosys.com/investors/reports-filings/quarterly-results.html

In [61]:
#q1_2024 = SimpleDirectoryReader(
#    input_files=["/content/ifrs-inr-press-release.pdf"]
#).load_data()

q1_2024 = SimpleDirectoryReader(
    input_files=["/content/output_documents/converted_document.md"]
).load_data()

DEBUG:fsspec.local:open file: /content/output_documents/converted_document.md


# Build indices

In [62]:
# dimensions of text-ada-embedding-002
d = 384
faiss_index = faiss.IndexFlatL2(d)

In [63]:
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
q1_2024_index = VectorStoreIndex.from_documents(q1_2024, storage_context=storage_context)

DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: ssojus

<!-- image -->

\

Solid Q1 year on yea...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: Key highlights:

## For the quarter ended June ...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: - bp recently signed an MoU with Infosys to dem...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: In Infosys, vidaXL has found a partner that pro...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: - Infosys Finacle helped successfully transform...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: 'Client Business Impact - FSI (APAC)', and 'Cli...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: UK, Nordics, Germany and Brazil
- Infosys recog...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: The risks and uncertainties relating to these s...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: 262           | -                |

## Build query engines

In [64]:
q1_2024__engine = q1_2024_index.as_query_engine(similarity_top_k=2)

In [65]:
# Assuming you have already built the index and query engine
retriever = q1_2024__engine.retriever
retrieved_nodes = retriever.retrieve("What is the revenue growth for the quarter?")

# Print the retrieved chunks
for i, node in enumerate(retrieved_nodes):
    print(f"\n--- Chunk {i+1} ---")
    print(node.get_text())

DEBUG:llama_index.core.indices.utils:> Top 2 nodes:
> [Node 9] [Similarity score:                 0.591031] 828          | 75,795           |
| Total liabilities and equity                               | ...
> [Node 0] [Similarity score:                 0.623795] ssojus

<!-- image -->

\

Solid Q1 year on year revenue growth of 4.2% at 20.8% operating margin...

--- Chunk 1 ---
828          | 75,795           |
| Total liabilities and equity                               | 131,322         | 125,816          |

## Extracted from the Condensed Consolidated statement of Comprehensive Income under IFRS for:

(In ` crore except per equity share data)

|                                       | 3 months ended June 30, 2023   | 3 months ended June 30, 2022   |
|---------------------------------------|--------------------------------|--------------------------------|
| Revenues                              | 37,933                         | 34,470                         |
| Cost of sales    

## Run queries

In [66]:
response = q1_2024__engine.query(
    "What is the revenue growth for the quarter?"
)

DEBUG:llama_index.core.indices.utils:> Top 2 nodes:
> [Node 9] [Similarity score:                 0.591031] 828          | 75,795           |
| Total liabilities and equity                               | ...
> [Node 0] [Similarity score:                 0.623795] ssojus

<!-- image -->

\

Solid Q1 year on year revenue growth of 4.2% at 20.8% operating margin...
DEBUG:asyncio:Using selector: EpollSelector
DEBUG:asyncio:Using selector: EpollSelector
DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-9a8a7886-3aad-4756-822c-b4f32e82f830', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'role': 'system', 'content': "You are an expert Q&A system that is trusted around the world.\nAlways answer the query using the provided context information, and not prior knowledge.\nSome rules to follow:\n1. Never directly reference the given context in your answer.\n2. Avoid 

In [67]:
print(response)

The quarter’s revenue grew by 4.2 percent year‑over‑year.


## Checking Prompt Templates

In [68]:
prompts_dict = q1_2024__engine.get_prompts()

In [69]:
# 1. see what's there
for k, v in prompts_dict.items():
    print(k, "\n", v.get_template(), "\n")

response_synthesizer:text_qa_template 
 Context information is below.
---------------------
{context_str}
---------------------
Given the context information and not prior knowledge, answer the query.
Query: {query_str}
Answer:  

response_synthesizer:refine_template 
 The original query is as follows: {query_str}
We have provided an existing answer: {existing_answer}
We have the opportunity to refine the existing answer (only if needed) with some more context below.
------------
{context_msg}
------------
Given the new context, refine the original answer to better answer the query. If the context isn't useful, return the original answer.
Refined Answer:  

response_synthesizer:chat_content_qa_template 
 
{% chat role="system" %}
You are an expert Q&A system that is trusted around the world.
Always answer the query using the provided context information, and not prior knowledge.
Some rules to follow:
1. Never directly reference the given context in your answer.
2. Avoid statements like

In [20]:
response = q1_2024__engine.query("Which are some of the key customer wins?")

In [21]:
print(response)

Some of the notable customer wins include:

- Enabling XacBank’s digital transformation with the Finacle Digital Banking Suite, providing a robust foundation for the bank’s growth strategy in Mongolia.  
- Partnering with vidaXL to transition eight agile development teams in under four months, establishing a comprehensive technology‑stack steering mechanism.  
- Extending collaboration with LexisNexis to deliver end‑to‑end information services across its content, enterprise, and product applications.  
- Working with the ATP to launch a digital Carbon Tracker that helps players monitor and reduce travel‑related carbon emissions.  
- Supporting Keytrade Bank in modernizing its core banking system, strengthening its foundation and future‑proofing the organization.


In [ ]:
response = q1_2024__engine.query("What are some of the key achievements in the quarter?")

In [ ]:
print(response)

In [ ]:
response = q1_2024__engine.query("What is the total assets?")

In [ ]:
print(response)

## Using your own prompt template

In [70]:
from llama_index.core import PromptTemplate

qa_tmpl = PromptTemplate(
    "You are a helpful assistant for answering finacial questions.\n"
    "Answer ONLY from the context below. If the answer is not in the context, say 'Not found'.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Question: {query_str}\n"
    "Answer: "
)

# 3. apply it
q1_2024__engine.update_prompts(
    {"response_synthesizer:text_qa_template": qa_tmpl}
)

In [71]:
response = q1_2024__engine.query("Which are some of the key customer wins?")

DEBUG:llama_index.core.indices.utils:> Top 2 nodes:
> [Node 4] [Similarity score:                 0.786872] - Infosys Finacle helped successfully transform XacBank's technology landscape with Finacle Digit...
> [Node 3] [Similarity score:                 0.8192] In Infosys, vidaXL has found a partner that proved to be dedicated in delivering this by transiti...
DEBUG:asyncio:Using selector: EpollSelector
DEBUG:asyncio:Using selector: EpollSelector
DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-c35406bd-5056-4b32-8400-c30f291f455b', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'role': 'user', 'content': 'You are a helpful assistant for answering finacial questions.\nAnswer ONLY from the context below. If the answer is not in the context, say \'Not found\'.\n---------------------\nfile_path: /content/output_documents/converted_document.md\n\n- Infosys Fi

In [72]:
print(response)

- XacBank – transformed its technology landscape with the Finacle Digital Banking Suite.  
- vidaXL – partnered with Infosys to transition eight agile development teams in less than four months.  
- LexisNexis – extended its collaboration with Infosys for end‑to‑end information services.  
- ATP – worked with Infosys to launch a digital Carbon Tracker for players.  
- Keytrade Bank – selected Infosys Finacle as the preferred partner to modernize its core banking system.
